# Spartan Search

Spartan crawler + SQLite FTS5 search.

- binary/file downloads are skipped before fetching
- non-text MIME bodies are not downloaded
- huge archive roots are skipped
- one host cannot consume the whole crawl budget
- same host: one connection at a time + delay


In [ ]:
from __future__ import annotations
import posixpath, socket, sqlite3, time
from collections import deque
from concurrent.futures import FIRST_COMPLETED, ThreadPoolExecutor, wait
from dataclasses import dataclass
from datetime import datetime, timezone
from urllib.parse import quote, unquote, urlsplit, urlunsplit

DEFAULT_PORT = 300
DB_PATH = "spartan.db"
TEXT_MIMES = {"text/gemini", "text/plain"}
MAX_BYTES_DEFAULT = 256 * 1024
MAX_PAGES_PER_HOST_DEFAULT = 250

BAD_SUFFIXES = (
    ".jpg",".jpeg",".png",".gif",".webp",".avif",".bmp",".ico",".svg",
    ".mp3",".ogg",".opus",".wav",".flac",".m4a",".mp4",".mkv",".webm",".mov",".avi",".m4v",
    ".pdf",".epub",".mobi",".zip",".tar",".gz",".bz2",".xz",".7z",".rar",".tgz",".tbz",".tbz2",
    ".iso",".img",".dmg",".exe",".bin",".apk",".deb",".rpm",".wasm",
    ".woff",".woff2",".ttf",".otf",".doc",".docx",".xls",".xlsx",".ppt",".pptx",
)
BLOCKED_PATH_PARTS = {"files","downloads","download","multimedia"}
ARCHIVE_ROOTS = {"rfc","man","mirrors","archive","archives","textfiles","gitrepositories"}

@dataclass(slots=True)
class Response:
    status: int
    meta: str
    body: bytes

def connect_db(path=DB_PATH):
    db = sqlite3.connect(path)
    db.row_factory = sqlite3.Row
    db.executescript("""
    PRAGMA journal_mode=WAL;
    CREATE TABLE IF NOT EXISTS pages(
      url TEXT PRIMARY KEY,title TEXT NOT NULL DEFAULT '',mime TEXT NOT NULL DEFAULT '',
      body TEXT NOT NULL DEFAULT '',status INTEGER NOT NULL DEFAULT 0,
      error TEXT NOT NULL DEFAULT '',fetched_at TEXT NOT NULL);
    CREATE TABLE IF NOT EXISTS links(
      source TEXT NOT NULL,target TEXT NOT NULL,PRIMARY KEY(source,target));
    CREATE INDEX IF NOT EXISTS links_target_idx ON links(target);
    """)
    try:
        db.execute("CREATE VIRTUAL TABLE IF NOT EXISTS pages_fts USING fts5(url UNINDEXED,title,body,tokenize='trigram')")
    except sqlite3.OperationalError:
        db.execute("CREATE VIRTUAL TABLE IF NOT EXISTS pages_fts USING fts5(url UNINDEXED,title,body,tokenize='unicode61')")
    return db

def is_excluded_url(url):
    p = urlsplit(url)
    path = unquote(p.path or "/")
    low = path.lower()
    if p.hostname in {"mozz.us","www.mozz.us"} and (low == "/ufo" or low.startswith("/ufo/")):
        return True
    parts = [x.lower() for x in path.split("/") if x]
    if set(parts) & BLOCKED_PATH_PARTS:
        return True
    if parts and parts[0] in ARCHIVE_ROOTS:
        return True
    return low.endswith(BAD_SUFFIXES)

def normalize_url(url):
    p = urlsplit(url.strip())
    if p.scheme.lower() != "spartan" or not p.hostname or p.query:
        return None
    try:
        host = p.hostname.encode("idna").decode("ascii").lower()
    except UnicodeError:
        return None
    path = quote(p.path or "/", safe="/%:@!$&'()*+,;=-._~")
    path = "/" + posixpath.normpath(path).lstrip("/")
    if (p.path or "/").endswith("/") and not path.endswith("/"):
        path += "/"
    netloc = host if p.port in (None, DEFAULT_PORT) else f"{host}:{p.port}"
    out = urlunsplit(("spartan",netloc,path,"",""))
    return None if is_excluded_url(out) else out

def resolve_url(base,target):
    target = target.strip()
    if not target or target.startswith(("#","=:","mailto:")):
        return None
    p = urlsplit(target)
    if p.scheme:
        return normalize_url(target)
    b = urlsplit(base)
    if target.startswith("//"):
        return normalize_url("spartan:" + target)
    if target.startswith("/"):
        path = target
    else:
        d = b.path if b.path.endswith("/") else posixpath.dirname(b.path) + "/"
        path = posixpath.join(d,target)
    return normalize_url(urlunsplit(("spartan",b.netloc,path,"","")))

def fetch_spartan(url,timeout=10.0,max_bytes=MAX_BYTES_DEFAULT):
    p = urlsplit(url)
    host = p.hostname.encode("idna").decode("ascii")
    request = f"{host} {p.path or '/'} 0\r\n".encode("ascii")
    with socket.create_connection((host,p.port or DEFAULT_PORT),timeout=timeout) as sock:
        sock.settimeout(timeout)
        sock.sendall(request)
        stream = sock.makefile("rb")
        line = stream.readline(4097)
        if not line or len(line) > 4096 or not line.endswith(b"\n"):
            raise OSError("invalid Spartan response")
        line = line.rstrip(b"\r\n")
        if not line or line[:1] not in b"2345":
            raise OSError(f"invalid Spartan status: {line[:32]!r}")
        status = int(chr(line[0]))
        meta = line[1:].lstrip(b" " ).decode("utf-8","replace")
        if status != 2:
            return Response(status,meta,b"")
        mime = meta.split(";",1)[0].strip().lower()
        if mime not in TEXT_MIMES:
            return Response(status,meta,b"")
        body = stream.read(max_bytes + 1)
        if len(body) > max_bytes:
            raise OSError(f"response exceeds {max_bytes} bytes")
        return Response(status,meta,body)

def parse_text(body,meta):
    mime = meta.split(";",1)[0].strip().lower() or "application/octet-stream"
    charset = "utf-8"
    for item in meta.split(";")[1:]:
        k,sep,v = item.strip().partition("=")
        if sep and k.lower() == "charset":
            charset = v.strip().strip('\"')
    try:
        return mime,body.decode(charset,"replace")
    except LookupError:
        return mime,body.decode("utf-8","replace")

def parse_gemtext(text,base):
    title, searchable, links, in_pre = "", [], [], False
    for raw in text.splitlines():
        if raw.startswith("```"):
            in_pre = not in_pre; continue
        if in_pre:
            searchable.append(raw); continue
        if raw.startswith("=:"):
            continue
        if raw.startswith("=>"):
            rest = raw[2:].strip()
            if rest:
                target,_,label = rest.partition(" " )
                u = resolve_url(base,target)
                if u: links.append(u)
                if label.strip(): searchable.append(label.strip())
            continue
        line = raw
        if raw.startswith("#"):
            line = raw.lstrip("#").strip()
            if not title and line: title = line
        elif raw.startswith(("*",">")):
            line = raw[1:].strip()
        if line.strip(): searchable.append(line.strip())
    return title,"\n".join(searchable),links

def save_page(db,url,title,mime,body,status,error,links):
    db.execute("""INSERT INTO pages(url,title,mime,body,status,error,fetched_at)
    VALUES(?,?,?,?,?,?,?) ON CONFLICT(url) DO UPDATE SET
    title=excluded.title,mime=excluded.mime,body=excluded.body,status=excluded.status,
    error=excluded.error,fetched_at=excluded.fetched_at""",
    (url,title,mime,body,status,error,datetime.now(timezone.utc).isoformat()))
    db.execute("DELETE FROM pages_fts WHERE url=?",(url,))
    if status == 2 and body:
        db.execute("INSERT INTO pages_fts(url,title,body) VALUES(?,?,?)",(url,title,body))
    db.execute("DELETE FROM links WHERE source=?",(url,))
    db.executemany("INSERT OR IGNORE INTO links(source,target) VALUES(?,?)",((url,x) for x in links))
    db.commit()

def purge_excluded(db):
    urls = {r[0] for r in db.execute("SELECT url FROM pages")}
    urls |= {r[0] for r in db.execute("SELECT source FROM links")}
    urls |= {r[0] for r in db.execute("SELECT target FROM links")}
    bad = [u for u in urls if is_excluded_url(u)]
    db.executemany("DELETE FROM pages_fts WHERE url=?",((u,) for u in bad))
    db.executemany("DELETE FROM pages WHERE url=?",((u,) for u in bad))
    db.executemany("DELETE FROM links WHERE source=? OR target=?",((u,u) for u in bad))
    db.commit()
    return len(bad)

def crawl(seeds,max_pages=500,delay=1.0,timeout=10.0,max_bytes=MAX_BYTES_DEFAULT,
          workers=8,max_pages_per_host=MAX_PAGES_PER_HOST_DEFAULT,db_path=DB_PATH):
    db = connect_db(db_path)
    removed = purge_excluded(db)
    if removed: print(f"purged {removed} excluded URLs")
    queue = deque(filter(None,(normalize_url(x) for x in seeds)))
    seen, active, next_allowed, per_host = set(queue), set(), {}, {}
    futures, scheduled, completed = {}, 0, 0

    with ThreadPoolExecutor(max_workers=workers) as pool:
        while (queue or futures) and completed < max_pages:
            now = time.monotonic()
            rotations = len(queue)
            while queue and len(futures) < workers and scheduled < max_pages and rotations:
                url = queue.popleft(); rotations -= 1
                host = urlsplit(url).netloc.lower()
                if per_host.get(host,0) >= max_pages_per_host:
                    continue
                if host in active or now < next_allowed.get(host,0):
                    queue.append(url); continue
                scheduled += 1
                per_host[host] = per_host.get(host,0) + 1
                print(f"[{scheduled}/{max_pages}] {url}")
                f = pool.submit(fetch_spartan,url,timeout,max_bytes)
                futures[f] = (url,host,time.monotonic())
                active.add(host); next_allowed[host] = time.monotonic() + delay

            if not futures:
                if queue: time.sleep(.05)
                continue
            done,_ = wait(futures,timeout=.1,return_when=FIRST_COMPLETED)
            for f in done:
                url,host,started = futures.pop(f)
                active.discard(host); completed += 1; links = []
                try:
                    r = f.result(); p = urlsplit(url)
                    if r.status == 3:
                        redir = resolve_url(url,r.meta)
                        save_page(db,url,"","","",3,f"redirect: {r.meta}",[])
                        if redir and urlsplit(redir).netloc == p.netloc and redir not in seen:
                            queue.append(redir); seen.add(redir)
                    elif r.status == 2:
                        mime,text = parse_text(r.body,r.meta)
                        if mime == "text/gemini":
                            title,body,links = parse_gemtext(text,url)
                            save_page(db,url,title,mime,body,2,"",links)
                        elif mime == "text/plain":
                            title = next((x.strip() for x in text.splitlines() if x.strip()),"")[:200]
                            save_page(db,url,title,mime,text,2,"",[])
                        else:
                            save_page(db,url,"",mime,"",2,"",[])
                    else:
                        save_page(db,url,"","","",r.status,r.meta,[])
                except Exception as e:
                    save_page(db,url,"","","",0,str(e),[])
                    print(f"  ! {url} — {e}")
                for link in links:
                    if link not in seen:
                        queue.append(link); seen.add(link)
                elapsed = time.monotonic() - started
                if elapsed >= max(2.0,timeout*.5):
                    print(f"  ↳ slow {elapsed:.1f}s: {url}")
    db.close()
    print(f"done: completed={completed}, discovered={len(seen)}, queued={len(queue)}, hosts={len(per_host)}, db={db_path}")


In [ ]:
SEEDS = ["spartan://spartan.mozz.us/"]
MAX_PAGES = 5000
WORKERS = 8
DELAY = 1.0
TIMEOUT = 10.0
MAX_BYTES = 256 * 1024
MAX_PAGES_PER_HOST = 250

# crawl(SEEDS, max_pages=MAX_PAGES, delay=DELAY, timeout=TIMEOUT,
#       max_bytes=MAX_BYTES, workers=WORKERS, max_pages_per_host=MAX_PAGES_PER_HOST)


In [ ]:
def search(query,limit=20,db_path=DB_PATH):
    query = query.strip()
    if not query: return []
    with connect_db(db_path) as db:
        if len(query) < 3:
            pat=f"%{query}%"
            return db.execute("SELECT url,title,body,0.0 score FROM pages WHERE status=2 AND (title LIKE ? OR body LIKE ?) LIMIT ?",(pat,pat,limit)).fetchall()
        try:
            match=" AND ".join(f'\"{x.replace(chr(34),chr(34)*2)}\"' for x in query.split())
            return db.execute("SELECT url,title,body,bm25(pages_fts,0.0,4.0,1.0) score FROM pages_fts WHERE pages_fts MATCH ? ORDER BY score LIMIT ?",(match,limit)).fetchall()
        except sqlite3.OperationalError:
            pat=f"%{query}%"
            return db.execute("SELECT url,title,body,0.0 score FROM pages WHERE status=2 AND (title LIKE ? OR body LIKE ?) LIMIT ?",(pat,pat,limit)).fetchall()

def print_results(query,limit=20):
    rows=search(query,limit)
    for i,r in enumerate(rows,1):
        flat=" ".join(r["body"].split())
        p=flat.casefold().find(query.casefold())
        start=max(0,p-60) if p>=0 else 0
        print(f"{i}. {r['title'] or '(untitled)'}")
        print("  ",r["url"])
        print("  ",("…" if start else "")+flat[start:start+180]+("…" if start+180<len(flat) else ""))
        print()
    print(f"{len(rows)} result(s)")

# Cleanup an existing DB once after updating this notebook:
# with connect_db() as db:
#     print("purged:", purge_excluded(db))
